In [ ]:
%pip install arch -q

In [ ]:
import numpy as np
import pandas as pd
import yfinance as yf

TICKERS = {
    "SP500": "^GSPC",
    "NASDAQ100": "^NDX",
    "DJIA": "^DJI",
}

raw_data = {}

for name, ticker in TICKERS.items():
    df = yf.download(
        ticker,
        start="2000-01-01",
        end="2025-09-01",
        auto_adjust=False,
        actions=False,
        progress=False,
    )

    if df.empty:
        raise ValueError(f"No data returned for {name} ({ticker}).")

    if isinstance(df.columns, pd.MultiIndex):
        df = df.xs(ticker, axis=1, level="Ticker")

    df = df[["Open", "High", "Low", "Close"]].copy()
    df.index = pd.to_datetime(df.index).tz_localize(None)
    df = df.loc[~df.index.duplicated(keep="first")].sort_index()

    valid = (
        np.isfinite(df).all(axis=1)
        & df.gt(0).all(axis=1)
        & df["High"].ge(df[["Open", "Close", "Low"]].max(axis=1))
        & df["Low"].le(df[["Open", "Close", "High"]].min(axis=1))
    )

    print(f"{name}: removing {(~valid).sum()} invalid OHLC rows")
    raw_data[name] = df.loc[valid].copy()

In [ ]:
common_dates = raw_data["SP500"].index

for df in raw_data.values():
    common_dates = common_dates.intersection(df.index)

common_dates = common_dates.sort_values()

if common_dates.empty:
    raise ValueError("No common trading dates across the three indices.")

data = {
    name: df.loc[common_dates].copy()
    for name, df in raw_data.items()
}

summary = pd.DataFrame([
    {
        "Index": name,
        "Ticker": TICKERS[name],
        "Observations": len(df),
        "Start": df.index.min().date(),
        "End": df.index.max().date(),
        "Removed for alignment": len(raw_data[name]) - len(df),
    }
    for name, df in data.items()
]).set_index("Index")

display(summary)
display(data["SP500"].head())

In [ ]:
YZ_WINDOW = 22
VARIANCE_FLOOR = 1e-12

def build_variance_features(ohlc, yz_window=YZ_WINDOW):
    if yz_window < 2:
        raise ValueError("Yang–Zhang requires at least two observations.")

    open_price = ohlc["Open"]
    high_price = ohlc["High"]
    low_price = ohlc["Low"]
    close_price = ohlc["Close"]

    log_return = np.log(close_price / close_price.shift(1))
    overnight_return = np.log(open_price / close_price.shift(1))
    intraday_return = np.log(close_price / open_price)

    close_to_close = log_return.pow(2)
    parkinson = np.log(high_price / low_price).pow(2) / (4 * np.log(2))

    rogers_satchell = (
        np.log(high_price / open_price) * np.log(high_price / close_price)
        + np.log(low_price / open_price) * np.log(low_price / close_price)
    )

    k = 0.34 / (1.34 + (yz_window + 1) / (yz_window - 1))

    yang_zhang = (
        overnight_return.rolling(yz_window).var(ddof=1)
        + k * intraday_return.rolling(yz_window).var(ddof=1)
        + (1 - k) * rogers_satchell.rolling(yz_window).mean()
    )

    features = pd.DataFrame({
        "log_return": log_return,
        "RV_CC": close_to_close,
        "RV_Parkinson": parkinson,
        "RV_YangZhang": yang_zhang,
    }).dropna()

    variance_columns = ["RV_CC", "RV_Parkinson", "RV_YangZhang"]

    if not np.isfinite(features.to_numpy()).all():
        raise ValueError("Non-finite values in variance features.")

    if features[variance_columns].lt(0).any().any():
        raise ValueError("Negative variance detected; inspect the OHLC data.")

    for column in variance_columns:
        features[f"log_{column}"] = np.log(
            features[column].clip(lower=VARIANCE_FLOOR)
        )

    return features

rv_data = {
    name: build_variance_features(ohlc)
    for name, ohlc in data.items()
}

display(rv_data["SP500"].head())

In [ ]:
YZ_WINDOW = 22
VARIANCE_FLOOR = 1e-12

def build_variance_features(ohlc, yz_window=YZ_WINDOW):
    if yz_window < 2:
        raise ValueError("Yang–Zhang requires at least two observations.")

    open_price = ohlc["Open"]
    high_price = ohlc["High"]
    low_price = ohlc["Low"]
    close_price = ohlc["Close"]

    log_return = np.log(close_price / close_price.shift(1))
    overnight_return = np.log(open_price / close_price.shift(1))
    intraday_return = np.log(close_price / open_price)

    close_to_close = log_return.pow(2)
    parkinson = np.log(high_price / low_price).pow(2) / (4 * np.log(2))

    rogers_satchell = (
        np.log(high_price / open_price) * np.log(high_price / close_price)
        + np.log(low_price / open_price) * np.log(low_price / close_price)
    )

    k = 0.34 / (1.34 + (yz_window + 1) / (yz_window - 1))

    yang_zhang = (
        overnight_return.rolling(yz_window).var(ddof=1)
        + k * intraday_return.rolling(yz_window).var(ddof=1)
        + (1 - k) * rogers_satchell.rolling(yz_window).mean()
    )

    features = pd.DataFrame({
        "log_return": log_return,
        "RV_CC": close_to_close,
        "RV_Parkinson": parkinson,
        "RV_YangZhang": yang_zhang,
    }).dropna()

    variance_columns = ["RV_CC", "RV_Parkinson", "RV_YangZhang"]

    if not np.isfinite(features.to_numpy()).all():
        raise ValueError("Non-finite values in variance features.")

    if features[variance_columns].lt(0).any().any():
        raise ValueError("Negative variance detected; inspect the OHLC data.")

    for column in variance_columns:
        features[f"log_{column}"] = np.log(
            features[column].clip(lower=VARIANCE_FLOOR)
        )

    return features

rv_data = {
    name: build_variance_features(ohlc)
    for name, ohlc in data.items()
}

display(rv_data["SP500"].head())

In [ ]:
variance_summary = pd.concat(
    {
        name: frame[["RV_CC", "RV_Parkinson", "RV_YangZhang"]].describe().T
        for name, frame in rv_data.items()
    },
    names=["Index", "Estimator"],
)

display(variance_summary)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

for ax, (name, frame) in zip(axes, rv_data.items()):
    for estimator in ["CC", "Parkinson", "YangZhang"]:
        ax.plot(
            frame.index,
            frame[f"log_RV_{estimator}"],
            label=estimator,
            linewidth=0.7,
            alpha=0.75,
        )

    ax.set_title(name)
    ax.set_ylabel("Log daily variance")
    ax.legend(loc="upper right")

axes[-1].set_xlabel("Date")
fig.suptitle("Variance proxies: Yang–Zhang uses a trailing 22-day window")
plt.tight_layout()
plt.show()

In [ ]:
LOOKBACK = 120
HORIZONS = (1, 5, 22)
TEST_FRACTION = 0.20
INPUT_COLUMNS = ["log_return", "RV_CC", "RV_Parkinson", "RV_YangZhang"]

def make_split(features, test_fraction=TEST_FRACTION):
    split_index = int(len(features) * (1.0 - test_fraction))
    train = features.iloc[:split_index]
    test = features.iloc[split_index:]

    if train.empty or test.empty:
        raise ValueError("Split produced an empty sample.")

    means = train[INPUT_COLUMNS].mean()
    stds = train[INPUT_COLUMNS].std(ddof=0).replace(0.0, 1.0)

    scaled = (features[INPUT_COLUMNS] - means) / stds
    scaled = pd.DataFrame(
        scaled.to_numpy(),
        index=features.index,
        columns=INPUT_COLUMNS,
    )

    return train, test, scaled, means, stds, split_index

splits = {}
for name, features in rv_data.items():
    train, test, scaled, means, stds, split_index = make_split(features)
    splits[name] = {
        "train": train,
        "test": test,
        "scaled": scaled,
        "means": means,
        "stds": stds,
        "split_index": split_index,
    }
    print(
        f"{name}: train {train.index[0].date()} → {train.index[-1].date()} "
        f"({len(train)} rows) | "
        f"test {test.index[0].date()} → {test.index[-1].date()} "
        f"({len(test)} rows)"
)

In [ ]:
TARGET_COLUMNS = ["log_RV_CC", "log_RV_Parkinson", "log_RV_YangZhang"]

for name, split in splits.items():
    split["scaled"] = split["scaled"].join(
        rv_data[name][TARGET_COLUMNS],
        how="left",
        rsuffix="_duplicate",
    )
    split["scaled"] = split["scaled"][INPUT_COLUMNS + TARGET_COLUMNS]

def make_sequences(scaled, target_column, horizon, lookback=LOOKBACK):
    if not isinstance(horizon, (int, np.integer)) or horizon < 1:
        raise ValueError("Horizon must be a positive integer.")

    if not isinstance(lookback, (int, np.integer)) or lookback < 1:
        raise ValueError("Lookback must be a positive integer.")

    inputs = scaled[INPUT_COLUMNS].to_numpy(dtype=np.float64)
    targets = scaled[target_column].to_numpy(dtype=np.float64)

    if not np.isfinite(inputs).all() or not np.isfinite(targets).all():
        raise ValueError("Inputs or targets contain missing or infinite values.")

    n_sequences = len(scaled) - lookback - horizon + 1

    if n_sequences < 1:
        raise ValueError(
            f"Not enough rows for lookback={lookback}, horizon={horizon}."
        )

    X = np.empty(
        (n_sequences, lookback, len(INPUT_COLUMNS)),
        dtype=np.float64,
    )

    for i in range(n_sequences):
        X[i] = inputs[i : i + lookback]

    target_positions = np.arange(n_sequences) + lookback + horizon - 1
    y = targets[target_positions]
    target_dates = scaled.index[target_positions]

    assert len(X) == len(y) == len(target_dates)
    assert target_dates[-1] == scaled.index[-1]

    return X, y, pd.DatetimeIndex(target_dates)

sequence_sets = {
    (name, estimator, horizon): make_sequences(
        splits[name]["scaled"],
        f"log_RV_{estimator}",
        horizon,
    )
    for name in rv_data
    for estimator in ("CC", "Parkinson", "YangZhang")
    for horizon in HORIZONS
}

example_key = ("SP500", "CC", 1)
X_example, y_example, dates_example = sequence_sets[example_key]

print(f"Key {example_key}: X {X_example.shape}, y {y_example.shape}")
print(f"First target date: {dates_example[0].date()}, last: {dates_example[-1].date()}")

In [ ]:
REGISTRY = {}
TRAIN_MASKS = {}
TEST_MASKS = {}

def build_forecast_registry(index_name, estimator, horizon):
    split = splits[index_name]
    scaled = split["scaled"]
    _, y, target_dates = sequence_sets[(index_name, estimator, horizon)]

    if scaled.index.has_duplicates:
        raise ValueError(f"{index_name}: duplicate dates in the aligned calendar.")
    if not scaled.index.is_monotonic_increasing:
        raise ValueError(f"{index_name}: calendar is not sorted.")

    date_positions = pd.Series(np.arange(len(scaled)), index=scaled.index)
    target_positions = date_positions.loc[target_dates].to_numpy()
    cutoff_positions = target_positions - horizon
    cutoff_dates = scaled.index[cutoff_positions]

    if (cutoff_positions < 0).any():
        raise ValueError("Cutoff precedes the start of the sample.")
    if not (cutoff_positions + horizon == target_positions).all():
        raise ValueError(
            f"{index_name}/{estimator}/h={horizon}: "
            "target is not exactly h trading days after the cutoff."
        )

    target_column = scaled[f"log_RV_{estimator}"].to_numpy()
    if not np.allclose(y, target_column[target_positions]):
        raise ValueError(
            f"{index_name}/{estimator}/h={horizon}: targets do not match the source column."
        )

    train_cutoff = scaled.index[split["split_index"] - 1]
    test_mask = np.asarray(target_dates > train_cutoff)
    train_mask = ~test_mask

    registry = pd.DataFrame({
        "cutoff_date": cutoff_dates,
        "target_date": target_dates,
    })
    registry["is_test"] = test_mask

    if registry.loc[test_mask, "target_date"].le(train_cutoff).any():
        raise ValueError("Test targets leak into the training period.")

    return registry, train_mask, test_mask

for index_name in data:
    for estimator in ("CC", "Parkinson", "YangZhang"):
        for horizon in HORIZONS:
            key = (index_name, estimator, horizon)
            registry, train_mask, test_mask = build_forecast_registry(*key)
            REGISTRY[key] = registry
            TRAIN_MASKS[key] = train_mask
            TEST_MASKS[key] = test_mask

print(f"Built {len(REGISTRY)} registries; all cutoff → target alignments verified.")

In [ ]:
CANONICAL_TEST_TARGETS = {}

for index_name in data:
    for horizon in HORIZONS:
        estimators = ("CC", "Parkinson", "YangZhang")
        target_sets = [
            REGISTRY[(index_name, est, horizon)].loc[
                TEST_MASKS[(index_name, est, horizon)], "target_date"
            ]
            for est in estimators
        ]
        for est, target_set in zip(estimators, target_sets):
            if not target_set.equals(target_sets[0]):
                raise ValueError(
                    f"{index_name}/h={horizon}: test targets differ across estimators ({est})."
                )
        CANONICAL_TEST_TARGETS[(index_name, horizon)] = pd.DatetimeIndex(target_sets[0])

rows = []
for (index_name, estimator, horizon), registry in sorted(REGISTRY.items()):
    key = (index_name, estimator, horizon)
    test_rows = registry.loc[registry["is_test"]]
    rows.append({
        "Index": index_name,
        "Estimator": estimator,
        "Horizon": horizon,
        "Sequences": len(registry),
        "Train targets": int(TRAIN_MASKS[key].sum()),
        "Test targets": int(TEST_MASKS[key].sum()),
        "First test cutoff": test_rows["cutoff_date"].iloc[0].date(),
        "First test target": test_rows["target_date"].iloc[0].date(),
        "Last test target": registry["target_date"].iloc[-1].date(),
    })

alignment_summary = pd.DataFrame(rows)
display(alignment_summary)

example_registry = REGISTRY[("SP500", "CC", 5)]
example_positions = pd.Series(
    np.arange(len(splits["SP500"]["scaled"])),
    index=splits["SP500"]["scaled"].index,
)
gaps = (
    example_positions.loc[example_registry["target_date"]].to_numpy()
    - example_positions.loc[example_registry["cutoff_date"]].to_numpy()
)
print("Cutoff → target trading-day gaps (SP500/CC/h=5):", np.unique(gaps))

for index_name in data:
    first_test = CANONICAL_TEST_TARGETS[(index_name, 1)][0]
    print(f"{index_name}: first h=1 test target {first_test.date()} (paper: August 2020)")

In [ ]:
REFIT_EVERY = 22

classical_forecasts = {}
arima_orders = {}

print(f"Refit interval: {REFIT_EVERY} test observations (anchored, expanding window)")

In [ ]:
import statsmodels.api as sm

def har_design(rv_frame, estimator_key, horizon):
    rv = rv_frame[f"RV_{estimator_key}"]
    log_rv = rv_frame[f"log_RV_{estimator_key}"]

    design = pd.DataFrame({
        "RV_d": log_rv,
        "RV_w": np.log(rv.rolling(5).mean().clip(lower=VARIANCE_FLOOR)),
        "RV_m": np.log(rv.rolling(22).mean().clip(lower=VARIANCE_FLOOR)),
    })

    X_all = design.to_numpy()
    y_all = log_rv.to_numpy()
    dates = rv_frame.index
    n = len(rv_frame)

    reg_valid = np.isfinite(X_all).all(axis=1)

    target_positions = np.arange(horizon, n)
    reg_positions = target_positions - horizon
    keep = reg_valid[reg_positions]

    X_aligned = X_all[reg_positions[keep]]
    y_aligned = y_all[target_positions[keep]]
    target_dates = dates[target_positions[keep]]
    cutoff_dates = dates[reg_positions[keep]]

    return (
        sm.add_constant(X_aligned, has_constant="add"),
        y_aligned,
        target_dates,
        cutoff_dates,
    )

def har_forecast(index_name, estimator_key, horizon):
    X, y, target_dates, cutoff_dates = har_design(rv_data[index_name], estimator_key, horizon)
    X = sm.add_constant(X, has_constant="add")

    split_index = splits[index_name]["split_index"]
    train_cutoff = rv_data[index_name].index[split_index - 1]

    test_indices = np.where(target_dates > train_cutoff)[0]

    if not np.array_equal(
        target_dates[test_indices],
        CANONICAL_TEST_TARGETS[(index_name, horizon)],
    ):
        raise ValueError("HAR target dates deviate from the canonical test set.")

    preds_log = np.empty(len(test_indices))
    params = None

    for j, i in enumerate(test_indices):
        if params is None or j % REFIT_EVERY == 0:
            usable = target_dates.searchsorted(cutoff_dates[i], side="right")
            if usable < 100:
                raise ValueError("Too few usable HAR rows at the first anchor.")
            params = sm.OLS(y[:usable], X[:usable]).fit().params
        preds_log[j] = X[i] @ params

    return pd.DataFrame(
        {
            "forecast_logvar": preds_log,
            "forecast_var": np.exp(np.clip(preds_log, -50, 50)),
        },
        index=target_dates[test_indices],
    )

for index_name in data:
    for estimator in ("CC", "Parkinson", "YangZhang"):
        for horizon in HORIZONS:
            key = (index_name, estimator, "HAR", horizon)
            classical_forecasts[key] = har_forecast(index_name, estimator, horizon)
    print(f"{index_name}: HAR done")

In [ ]:
import warnings

def select_arima_order(series_values, max_p=3, max_d=1, max_q=3):
    best = None
    for d in range(max_d + 1):
        for p in range(max_p + 1):
            for q in range(max_q + 1):
                if p == 0 and q == 0:
                    continue
                try:
                    with warnings.catch_warnings():
                        warnings.simplefilter("ignore")
                        fit = sm.tsa.ARIMA(series_values, order=(p, d, q)).fit()
                except Exception:
                    continue
                if np.isfinite(fit.aic) and (best is None or fit.aic < best[0]):
                    best = (fit.aic, (p, d, q), fit)
    if best is None:
        raise RuntimeError("No ARIMA candidate converged.")
    return best

def arima_forecast(index_name, estimator_key, horizon, train_fit, series_values, date_index):
    registry = REGISTRY[(index_name, estimator_key, horizon)]
    test_rows = registry.loc[registry["is_test"]]

    if not np.array_equal(
        test_rows["target_date"].to_numpy(),
        CANONICAL_TEST_TARGETS[(index_name, horizon)].to_numpy(),
    ):
        raise ValueError("ARIMA target dates deviate from the canonical test set.")

    position_of = pd.Series(np.arange(len(date_index)), index=date_index)
    cutoff_positions = position_of.loc[test_rows["cutoff_date"]].to_numpy()

    preds_log = np.empty(len(test_rows))

    for j, pos in enumerate(cutoff_positions):
        history = series_values[: pos + 1]
        applied = train_fit.apply(history, refit=False)
        preds_log[j] = float(applied.forecast(steps=horizon)[-1])
        if j % 250 == 0:
            print(f"  {index_name}/{estimator_key}/h={horizon}: origin {j}/{len(test_rows)}")

    return pd.DataFrame(
        {
            "forecast_logvar": preds_log,
            "forecast_var": np.exp(np.clip(preds_log, -50, 50)),
        },
        index=test_rows["target_date"],
    )

for index_name in data:
    for estimator in ("CC", "Parkinson", "YangZhang"):
        scaled = splits[index_name]["scaled"]
        series_values = scaled[f"log_RV_{estimator}"].to_numpy()
        split_index = splits[index_name]["split_index"]

        aic, order, train_fit = select_arima_order(series_values[:split_index])
        arima_orders[(index_name, estimator)] = {"order": order, "AIC": aic}
        print(f"{index_name}/{estimator}: selected ARIMA{order} (AIC {aic:.1f})")

        for horizon in HORIZONS:
            key = (index_name, estimator, "ARIMA", horizon)
            classical_forecasts[key] = arima_forecast(
                index_name, estimator, horizon, train_fit, series_values, scaled.index
            )

    print(f"{index_name}: ARIMA done")

In [ ]:
from arch import arch_model

def garch_forecast_index(index_name, horizon):
    rv_index = rv_data[index_name].index
    log_ret = np.log(data[index_name]["Close"]).diff().reindex(rv_index)
    returns_pct = (100 * log_ret).dropna()

    if len(returns_pct) != len(rv_index):
        raise ValueError(f"{index_name}: return series misaligned with RV calendar.")

    split_index = splits[index_name]["split_index"]
    registry = REGISTRY[(index_name, "CC", horizon)]
    test_rows = registry.loc[registry["is_test"]]

    if not np.array_equal(
        test_rows["target_date"].to_numpy(),
        CANONICAL_TEST_TARGETS[(index_name, horizon)].to_numpy(),
    ):
        raise ValueError("GARCH target dates deviate from the canonical test set.")

    forecasts = pd.Series(np.nan, index=test_rows["target_date"])
    params = None

    for j, (cutoff, target) in enumerate(zip(test_rows["cutoff_date"], test_rows["target_date"])):
        history = returns_pct.loc[:cutoff]

        if params is None or j % REFIT_EVERY == 0:
            res = arch_model(
                history, mean="Constant", vol="GARCH", p=1, q=1, dist="normal"
            ).fit(disp="off")
            params = res.params

        fixed_res = arch_model(
            history, mean="Constant", vol="GARCH", p=1, q=1, dist="normal"
        ).fix(params)

        var_path = fixed_res.forecast(horizon=horizon, reindex=False).variance
        forecasts.loc[target] = float(var_path.iloc[-1, horizon - 1]) / 1e4

        if j % 250 == 0:
            print(f"  {index_name}/h={horizon}: origin {j}/{len(test_rows)}")

    return forecasts

for index_name in data:
    for horizon in HORIZONS:
        garch_var = garch_forecast_index(index_name, horizon)
        for estimator in ("CC", "Parkinson", "YangZhang"):
            key = (index_name, estimator, "GARCH", horizon)
            classical_forecasts[key] = pd.DataFrame({
                "forecast_logvar": np.log(garch_var.clip(lower=VARIANCE_FLOOR)),
                "forecast_var": garch_var,
            })
    print(f"{index_name}: GARCH done")

In [ ]:
if any(df["forecast_var"].isna().any() for df in classical_forecasts.values()):
    raise ValueError("NaN forecasts detected in classical results.")

expected_keys = {
    (i, e, m, h)
    for i in data
    for e in ("CC", "Parkinson", "YangZhang")
    for m in ("HAR", "ARIMA", "GARCH")
    for h in HORIZONS
}
if set(classical_forecasts) != expected_keys:
    raise ValueError("Classical forecast registry is incomplete.")

sanity_rows = []
for key, df in sorted(classical_forecasts.items()):
    index_name, estimator, model, horizon = key
    realized = np.exp(
        splits[index_name]["scaled"].loc[df.index, f"log_RV_{estimator}"]
    )
    sanity_rows.append({
        "Index": index_name,
        "Estimator": estimator,
        "Model": model,
        "Horizon": horizon,
        "Test obs": len(df),
        "Mean forecast RV": df["forecast_var"].mean(),
        "Mean realized RV": realized.mean(),
    })

sanity_table = pd.DataFrame(sanity_rows)
display(sanity_table)
display(pd.DataFrame(arima_orders).T)

In [ ]:
import math
import random
import time

import torch
import torch.nn as nn

SEED = 42
BATCH_SIZE = 64
LR = 1e-3
MAX_EPOCHS = 100
PATIENCE = 10
VAL_TAIL = 63
GRAD_CLIP = 1.0
DL_MODELS = ("LSTM", "CNN_LSTM", "Transformer", "PatchTST_lite")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def qlike_loss(pred, target):
    return (pred + torch.exp(torch.clamp(target - pred, max=30.0))).mean()

In [ ]:
def sinusoidal_pe(max_len, d_model):
    position = torch.arange(max_len).unsqueeze(1).float()
    div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
    pe = torch.zeros(max_len, d_model)
    pe[:, 0::2] = torch.sin(position * div)
    pe[:, 1::2] = torch.cos(position * div)
    return pe

class LSTMModel(nn.Module):
    def __init__(self, n_features, hidden=64, layers=2, dropout=0.1):
        super().__init__()
        self.lstm = nn.LSTM(n_features, hidden, layers, dropout=dropout, batch_first=True)
        self.head = nn.Linear(hidden, 1)

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.head(out[:, -1]).squeeze(-1)

class CNNLSTMModel(nn.Module):
    def __init__(self, n_features, conv_channels=(32, 64), hidden=64, dropout=0.1):
        super().__init__()
        c1, c2 = conv_channels
        self.conv = nn.Sequential(
            nn.Conv1d(n_features, c1, kernel_size=3, padding=1), nn.ReLU(),
            nn.Conv1d(c1, c2, kernel_size=3, padding=1), nn.ReLU(),
            nn.MaxPool1d(2),
        )
        self.lstm = nn.LSTM(c2, hidden, batch_first=True)
        self.dropout = nn.Dropout(dropout)
        self.head = nn.Linear(hidden, 1)

    def forward(self, x):
        z = self.conv(x.transpose(1, 2))
        out, _ = self.lstm(z.transpose(1, 2))
        return self.head(self.dropout(out[:, -1])).squeeze(-1)

class TransformerModel(nn.Module):
    def __init__(self, n_features, d_model=64, heads=4, ff=128, layers=2, dropout=0.1, max_len=512):
        super().__init__()
        self.input_proj = nn.Linear(n_features, d_model)
        self.register_buffer("pe", sinusoidal_pe(max_len, d_model))
        encoder_layer = nn.TransformerEncoderLayer(
            d_model, heads, ff, dropout, batch_first=True, norm_first=True
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, layers)
        self.head = nn.Linear(d_model, 1)

    def forward(self, x):
        h = self.input_proj(x) + self.pe[: x.size(1)]
        h = self.encoder(h)
        return self.head(h.mean(dim=1)).squeeze(-1)

class PatchTSTLite(nn.Module):
    def __init__(self, n_features, patch_len=12, stride=12, d_model=64, heads=4, ff=128, layers=3, dropout=0.1, max_len=512):
        super().__init__()
        self.patch_len = patch_len
        self.stride = stride
        self.embed = nn.Linear(patch_len, d_model)
        self.register_buffer("pe", sinusoidal_pe(max_len, d_model))
        encoder_layer = nn.TransformerEncoderLayer(
            d_model, heads, ff, dropout, batch_first=True, norm_first=True
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, layers)
        self.head = nn.Linear(d_model, 1)

    def forward(self, x):
        batch, time_steps, channels = x.shape
        n_patches = (time_steps - self.patch_len) // self.stride + 1
        if n_patches < 1:
            raise ValueError("Sequence too short for one patch.")
        patch_idx = (
            torch.arange(self.patch_len, device=x.device).unsqueeze(0)
            + self.stride * torch.arange(n_patches, device=x.device).unsqueeze(1)
        )
        patches = x[:, patch_idx, :]
        patches = patches.permute(0, 3, 1, 2).reshape(
            batch * channels, n_patches, self.patch_len
        )
        h = self.embed(patches) + self.pe[:n_patches]
        h = self.encoder(h)
        h = h.mean(dim=1).view(batch, channels, -1)
        return self.head(h).squeeze(-1).mean(dim=1)

def build_model(name, n_features):
    if name == "LSTM":
        return LSTMModel(n_features)
    if name == "CNN_LSTM":
        return CNNLSTMModel(n_features)
    if name == "Transformer":
        return TransformerModel(n_features)
    if name == "PatchTST_lite":
        return PatchTSTLite(n_features)
    raise ValueError(f"Unknown model {name}")

In [ ]:
def evaluate_qlike(model, X, y, batch_size=BATCH_SIZE):
    model.eval()
    total = 0.0
    with torch.no_grad():
        for start in range(0, len(X), batch_size):
            pred = model(X[start : start + batch_size])
            total += qlike_loss(pred, y[start : start + batch_size]).item() * len(pred)
    return total / len(X)

def train_model(model, X_train, y_train, X_val, y_val):
    opt = torch.optim.Adam(model.parameters(), lr=LR)
    best_val = float("inf")
    best_state = None
    best_epoch = 0
    wait = 0

    for epoch in range(MAX_EPOCHS):
        model.train()
        perm = torch.randperm(len(X_train), device=X_train.device)
        for start in range(0, len(perm), BATCH_SIZE):
            idx = perm[start : start + BATCH_SIZE]
            opt.zero_grad()
            loss = qlike_loss(model(X_train[idx]), y_train[idx])
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            opt.step()

        val_loss = evaluate_qlike(model, X_val, y_val)
        if val_loss < best_val - 1e-8:
            best_val = val_loss
            best_epoch = epoch + 1
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
            if wait >= PATIENCE:
                break

    if best_state is not None:
        model.load_state_dict(best_state)
    return {"best_epoch": best_epoch, "val_qlike": best_val}

def predict_logvar(model, X, batch_size=BATCH_SIZE):
    model.eval()
    preds = []
    with torch.no_grad():
        for start in range(0, len(X), batch_size):
            preds.append(model(X[start : start + batch_size]).cpu().numpy())
    return np.concatenate(preds)

In [ ]:
dl_forecasts = {}
dl_runs = []

def tensors_to_device(X, y):
    return (
        torch.tensor(X, dtype=torch.float32, device=device),
        torch.tensor(y, dtype=torch.float32, device=device),
    )

for index_name in data:
    for estimator in ("CC", "Parkinson", "YangZhang"):
        for horizon in HORIZONS:
            combo = (index_name, estimator, horizon)
            X, y, _ = sequence_sets[combo]
            train_mask = TRAIN_MASKS[combo]
            test_mask = TEST_MASKS[combo]

            registry = REGISTRY[combo]
            test_target_dates = pd.DatetimeIndex(
                registry.loc[registry["is_test"], "target_date"]
            )

            if len(test_target_dates) != int(test_mask.sum()):
                raise ValueError(f"{combo}: test mask does not match the registry.")
            if not np.array_equal(
                test_target_dates.to_numpy(),
                CANONICAL_TEST_TARGETS[(index_name, horizon)].to_numpy(),
            ):
                raise ValueError(f"{combo}: test targets deviate from the canonical set.")

            X_train_np, y_train_np = X[train_mask], y[train_mask]
            X_test_np = X[test_mask]

            split = len(X_train_np) - VAL_TAIL
            if split < 100:
                raise ValueError(f"{combo}: not enough training rows.")

            X_tr, y_tr = tensors_to_device(X_train_np[:split], y_train_np[:split])
            X_va, y_va = tensors_to_device(X_train_np[split:], y_train_np[split:])
            X_te, _ = tensors_to_device(X_test_np, y[test_mask])

            target_mean = float(y_train_np[:split].mean())

            for model_name in DL_MODELS:
                set_seed()
                model = build_model(model_name, X.shape[2]).to(device)
                model.head.bias.data.fill_(target_mean)

                started = time.time()
                info = train_model(model, X_tr, y_tr, X_va, y_va)
                train_qlike = evaluate_qlike(model, X_tr, y_tr)
                preds_log = predict_logvar(model, X_te)
                elapsed = time.time() - started

                key = (index_name, estimator, model_name, horizon)
                dl_forecasts[key] = pd.DataFrame(
                    {
                        "forecast_logvar": preds_log,
                        "forecast_var": np.exp(np.clip(preds_log, -50, 50)),
                    },
                    index=test_target_dates,
                )

                dl_runs.append({
                    "Index": index_name,
                    "Estimator": estimator,
                    "Horizon": horizon,
                    "Model": model_name,
                    "Train QLIKE": train_qlike,
                    "Best epoch": info["best_epoch"],
                    "Val QLIKE": info["val_qlike"],
                    "Seconds": round(elapsed, 1),
                    "Params": sum(p.numel() for p in model.parameters()),
                })
                print(f"{key}: best epoch {info['best_epoch']}, train QLIKE {train_qlike:.4f}, val QLIKE {info['val_qlike']:.4f}, {elapsed:.0f}s")

            print(f"{combo}: done")

In [ ]:
if any(df["forecast_var"].isna().any() for df in dl_forecasts.values()):
    raise ValueError("NaN forecasts detected in DL results.")

expected_keys = {
    (i, e, m, h)
    for i in data
    for e in ("CC", "Parkinson", "YangZhang")
    for m in DL_MODELS
    for h in HORIZONS
}
if set(dl_forecasts) != expected_keys:
    raise ValueError("DL forecast registry is incomplete.")

dl_runs_table = pd.DataFrame(dl_runs)
display(dl_runs_table)
print(f"Total training time: {dl_runs_table['Seconds'].sum() / 60:.1f} minutes")

In [ ]:
MODEL_ORDER = ("HAR", "ARIMA", "GARCH", "LSTM", "CNN_LSTM", "Transformer", "PatchTST_lite")

all_forecasts = {**classical_forecasts, **dl_forecasts}

test_losses = {}
results_rows = []

for combo in sequence_sets:
    index_name, estimator, horizon = combo
    target_dates = CANONICAL_TEST_TARGETS[(index_name, horizon)]

    y_all = sequence_sets[combo][1]
    realized_var = np.exp(y_all[TEST_MASKS[combo]])

    combo_forecasts = {
        key: df for key, df in all_forecasts.items()
        if key[0] == index_name and key[1] == estimator and key[3] == horizon
    }
    if len(combo_forecasts) != len(MODEL_ORDER):
        raise ValueError(f"{combo}: expected {len(MODEL_ORDER)} models, found {len(combo_forecasts)}.")

    for key, forecast_df in combo_forecasts.items():
        model = key[2]
        fv = forecast_df["forecast_var"].reindex(target_dates)
        if fv.isna().any():
            raise ValueError(f"{key}: forecasts do not cover the canonical test targets.")

        fv_clipped = fv.clip(lower=VARIANCE_FLOOR).to_numpy()
        qlike_t = np.log(fv_clipped) + realized_var / fv_clipped
        err = fv_clipped - realized_var

        if not np.isfinite(qlike_t).all():
            raise ValueError(f"{key}: non-finite QLIKE values.")

        test_losses[key] = pd.Series(qlike_t, index=target_dates)
        results_rows.append({
            "Index": index_name,
            "Estimator": estimator,
            "Horizon": horizon,
            "Model": model,
            "QLIKE": qlike_t.mean(),
            "RMSE": np.sqrt(np.mean(err ** 2)),
            "MAE": np.mean(np.abs(err)),
        })

results = pd.DataFrame(results_rows)
print(f"Evaluated {len(results)} (index × estimator × horizon × model) rows.")

In [ ]:
for index_name in data:
    panel = results[(results["Index"] == index_name) & (results["Estimator"] == "CC")]
    print(f"\n=== {index_name} — Close-to-Close: QLIKE by horizon ===")
    display(
        panel.pivot(index="Model", columns="Horizon", values="QLIKE")
        .reindex(MODEL_ORDER)
        .round(5)
    )

results["QLIKE_rank"] = results.groupby(["Index", "Estimator", "Horizon"])["QLIKE"].rank()

rank_table = (
    results.pivot_table(index="Model", columns="Horizon", values="QLIKE_rank", aggfunc="mean")
    .reindex(MODEL_ORDER)
    .round(2)
)
print("Mean QLIKE rank across all 27 combos (1 = best):")
display(rank_table)

winners = (
    results.loc[results.groupby(["Index", "Estimator", "Horizon"])["QLIKE"].idxmin(),
                ["Index", "Estimator", "Horizon", "Model", "QLIKE"]]
    .reset_index(drop=True)
)
display(winners)

In [ ]:
overfit = results[results["Model"].isin(DL_MODELS)].merge(
    dl_runs_table,
    on=["Index", "Estimator", "Horizon", "Model"],
    how="inner",
    validate="many_to_one",
)

overfit["Test − Train QLIKE"] = overfit["QLIKE"] - overfit["Train QLIKE"]
overfit["Test − Val QLIKE"] = overfit["QLIKE"] - overfit["Val QLIKE"]

if len(overfit) != len(dl_runs_table):
    raise ValueError("Overfitting merge did not match the DL run log.")

print("Mean test − train QLIKE gap (positive = in-sample advantage / overfitting):")
display(
    overfit.pivot_table(index="Model", columns="Horizon", values="Test − Train QLIKE", aggfunc="mean")
    .reindex(DL_MODELS)
    .round(5)
)

print("Mean test − validation QLIKE gap (positive = generalization loss vs early-stopping pick):")
display(
    overfit.pivot_table(index="Model", columns="Horizon", values="Test − Val QLIKE", aggfunc="mean")
    .reindex(DL_MODELS)
    .round(5)
)

print("Best-epoch distribution (early-stopping headroom):")
display(
    dl_runs_table.pivot_table(index="Model", columns="Horizon", values="Best epoch", aggfunc="median")
    .reindex(DL_MODELS)
)

In [ ]:
results.to_csv("results_summary.csv", index=False)
test_losses_frame = pd.DataFrame(test_losses)
test_losses_frame.to_csv("test_losses_per_date.csv")
print("Saved results_summary.csv and test_losses_per_date.csv to the working directory.")

In [ ]:
from scipy import stats

def dm_test(loss_model, loss_benchmark, hac_lag):
    d = (loss_model - loss_benchmark).dropna()
    if d.empty:
        raise ValueError("Empty loss differential.")
    n = len(d)
    dbar = d.mean()
    dc = d - dbar

    vlrv = float(np.mean(dc ** 2))
    for k in range(1, hac_lag + 1):
        gamma_k = float(np.mean(dc.to_numpy()[k:] * dc.to_numpy()[:-k]))
        vlrv += 2.0 * (1.0 - k / (hac_lag + 1)) * gamma_k

    vlrv /= n
    if vlrv <= 0:
        raise ValueError("Non-positive long-run variance; inspect the loss differential.")

    dm = dbar / np.sqrt(vlrv)
    p_two_sided = 2.0 * (1.0 - stats.norm.cdf(abs(dm)))
    return dm, p_two_sided, dbar, n

In [ ]:
BENCHMARK_MODEL = "HAR"
CLASSICAL_MODELS = ("HAR", "ARIMA", "GARCH")
CRITICAL = 1.96

dm_rows = []

for index_name in data:
    for estimator in ("CC", "Parkinson", "YangZhang"):
        for horizon in HORIZONS:
            panel = results[
                (results["Index"] == index_name)
                & (results["Estimator"] == estimator)
                & (results["Horizon"] == horizon)
                & (results["Model"].isin(CLASSICAL_MODELS))
            ]
            best_classical = panel.loc[panel["QLIKE"].idxmin(), "Model"]

            benchmarks = [BENCHMARK_MODEL]
            if best_classical != BENCHMARK_MODEL:
                benchmarks.append(best_classical)

            for benchmark in benchmarks:
                bench_loss = test_losses[(index_name, estimator, benchmark, horizon)]

                for model in MODEL_ORDER:
                    if model == benchmark:
                        continue

                    model_loss = test_losses[(index_name, estimator, model, horizon)]
                    dm_primary, p_primary, dbar, n_obs = dm_test(
                        model_loss, bench_loss, max(horizon - 1, 0)
                    )
                    dm_robust, p_robust, _, _ = dm_test(model_loss, bench_loss, horizon)

                    if p_primary < 0.05 and dbar < 0:
                        verdict = f"better than {benchmark} (5%)"
                    elif p_primary < 0.05 and dbar > 0:
                        verdict = f"worse than {benchmark} (5%)"
                    else:
                        verdict = "no significant difference"

                    dm_rows.append({
                        "Index": index_name,
                        "Estimator": estimator,
                        "Horizon": horizon,
                        "Benchmark": benchmark,
                        "Model": model,
                        "Mean loss diff": dbar,
                        "DM (lag h−1)": dm_primary,
                        "p (lag h−1)": p_primary,
                        "DM (lag h)": dm_robust,
                        "p (lag h)": p_robust,
                        "Test obs": n_obs,
                        "Verdict": verdict,
                    })

dm_table = pd.DataFrame(dm_rows)
print(f"Ran {len(dm_table)} DM tests.")

In [ ]:
primary = dm_table[dm_table["Benchmark"] == BENCHMARK_MODEL]

print(f"=== DM vs {BENCHMARK_MODEL}: SP500 / CC panel ===")
display(
    primary[(primary["Index"] == "SP500") & (primary["Estimator"] == "CC")]
    .drop(columns=["Benchmark"])
    .reset_index(drop=True)
    .round(4)
)

sig_better = primary[primary["p (lag h−1)"] < 0.05].loc[
    lambda df: df["Mean loss diff"] < 0
].groupby(["Model", "Horizon"]).size()
sig_worse = primary[primary["p (lag h−1)"] < 0.05].loc[
    lambda df: df["Mean loss diff"] > 0
].groupby(["Model", "Horizon"]).size()

summary_counts = pd.DataFrame({
    "Sig. better (of 9)": sig_better,
    "Sig. worse (of 9)": sig_worse,
}).fillna(0).astype(int)

print(f"\nSignificant DM outcomes vs {BENCHMARK_MODEL}, by model and horizon (9 combos = 3 indices × 3 estimators):")
display(summary_counts)

robust_agree = (np.sign(primary["DM (lag h−1)"]) == np.sign(primary["DM (lag h)"])) & (
    (primary["p (lag h)"] < 0.05) == (primary["p (lag h−1)"] < 0.05)
)
print(f"Lag-h robustness check agrees with lag-h−1 on {robust_agree.sum()}/{len(primary)} tests.")

In [ ]:
dm_table.to_csv("dm_tests.csv", index=False)
print("Saved dm_tests.csv to the working directory.")

In [ ]:
panel_tables = {}

for index_name in data:
    for estimator in ("CC", "Parkinson", "YangZhang"):
        panel = results[
            (results["Index"] == index_name) & (results["Estimator"] == estimator)
        ]
        table = (
            panel.pivot(index="Model", columns="Horizon", values="QLIKE")
            .reindex(MODEL_ORDER)
            .round(5)
        )
        rmse_table = (
            panel.pivot(index="Model", columns="Horizon", values="RMSE")
            .reindex(MODEL_ORDER)
            .round(5)
        )
        mae_table = (
            panel.pivot(index="Model", columns="Horizon", values="MAE")
            .reindex(MODEL_ORDER)
            .round(5)
        )
        panel_tables[(index_name, estimator)] = {
            "QLIKE": table, "RMSE": rmse_table, "MAE": mae_table
        }

        print(f"\n=== {index_name} / {estimator} — QLIKE (rows = model, cols = horizon) ===")
        display(table)

In [ ]:
results_full_export = pd.concat(
    {
        metric: pd.concat(
            {f"{i}/{e}": t[metric] for (i, e), t in panel_tables.items()},
            names=["Panel"],
        )
        for metric in ("QLIKE", "RMSE", "MAE")
    },
    names=["Metric"],
)
results_full_export.to_csv("results_panels.csv")
print("Saved results_panels.csv (QLIKE/RMSE/MAE for all 9 panels).")

In [ ]:
best_dl_by_index = {}
for index_name in data:
    dl_res = results[results["Model"].isin(DL_MODELS) & (results["Index"] == index_name)]
    mean_rank = dl_res.groupby("Model")["QLIKE_rank"].mean()
    best_dl_by_index[index_name] = mean_rank.idxmin()

fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

for ax, index_name in zip(axes, data):
    combo = (index_name, "CC", 1)
    dates = CANONICAL_TEST_TARGETS[(index_name, 1)]
    realized = sequence_sets[combo][1][TEST_MASKS[combo]]

    ax.plot(dates, realized, color="black", linewidth=0.7, label="Realized log-RV")
    ax.plot(
        dates,
        classical_forecasts[(index_name, "CC", "HAR", 1)]["forecast_logvar"],
        linewidth=0.7, alpha=0.8, label="HAR",
    )
    best_dl = best_dl_by_index[index_name]
    ax.plot(
        dates,
        dl_forecasts[(index_name, "CC", best_dl, 1)]["forecast_logvar"],
        linewidth=0.7, alpha=0.8, label=f"Best DL: {best_dl}",
    )

    ax.set_title(index_name)
    ax.set_ylabel("Log daily variance")
    ax.legend(loc="upper right", fontsize=8)

axes[-1].set_xlabel("Date")
fig.suptitle("Test-period forecasts vs realized log-RV (CC, h = 1)")
plt.tight_layout()
plt.savefig("fig_predicted_vs_realized.png", dpi=150)
plt.show()
print("Best DL model per index (mean QLIKE rank over 9 combos):", best_dl_by_index)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5), sharey=False)

for ax, index_name in zip(axes, data):
    panel = results[(results["Index"] == index_name) & (results["Estimator"] == "CC")]
    bar_data = (
        panel.pivot(index="Model", columns="Horizon", values="QLIKE")
        .reindex(MODEL_ORDER)
    )
    x = np.arange(len(MODEL_ORDER))
    width = 0.26
    for k, horizon in enumerate(HORIZONS):
        ax.bar(x + (k - 1) * width, bar_data[horizon], width, label=f"h = {horizon}")

    ax.set_xticks(x)
    ax.set_xticklabels(MODEL_ORDER, rotation=45, ha="right", fontsize=8)
    ax.set_title(index_name)
    ax.set_ylabel("QLIKE (test, CC)")

axes[0].legend(fontsize=8)
fig.suptitle("Forecast accuracy by model and horizon (CC estimator)")
plt.tight_layout()
plt.savefig("fig_qlike_bars.png", dpi=150)
plt.show()

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(14, 10), sharex=False)

for row, index_name in enumerate(data):
    for col, horizon in enumerate((1, 22)):
        ax = axes[row, col]
        for model in MODEL_ORDER:
            if model == "HAR":
                continue
            diff = test_losses[(index_name, "CC", model, horizon)] - test_losses[
                (index_name, "CC", "HAR", horizon)
            ]
            ax.plot(diff.index, diff.cumsum(), linewidth=0.9, label=model)

        ax.axhline(0, color="black", linewidth=0.6, linestyle="--")
        ax.set_title(f"{index_name}, h = {horizon}", fontsize=9)
        ax.set_ylabel("Cumulative ΔQLIKE vs HAR")

axes[0, 0].legend(fontsize=7, loc="upper left")
fig.suptitle("Cumulative loss differential vs HAR (below zero = model outperforms)")
plt.tight_layout()
plt.savefig("fig_cumulative_loss_diff.png", dpi=150)
plt.show()

In [ ]:
frames = []

for combo in sequence_sets:
    index_name, estimator, horizon = combo
    dates = CANONICAL_TEST_TARGETS[(index_name, horizon)]

    realized_logvar = sequence_sets[combo][1][TEST_MASKS[combo]]
    base = pd.DataFrame(
        {
            "realized_logvar": realized_logvar,
            "realized_var": np.exp(realized_logvar),
        },
        index=dates,
    )

    for model in MODEL_ORDER:
        part = all_forecasts[(index_name, estimator, model, horizon)].join(base)
        fv = part["forecast_var"].clip(lower=VARIANCE_FLOOR)
        part["QLIKE_t"] = np.log(fv) + part["realized_var"] / fv
        part.insert(0, "Horizon", horizon)
        part.insert(0, "Estimator", estimator)
        part.insert(0, "Index", index_name)
        part.insert(0, "Model", model)
        frames.append(part.reset_index().rename(columns={"index": "target_date"}))

forecast_dump = pd.concat(frames, ignore_index=True)
forecast_dump.to_csv("forecast_dump.csv", index=False)

print("=== RUN RECAP ===")
print(f"Forecast rows: {len(forecast_dump):,} (expect {27 * 7 * {1: 1}[1]}...)")
print(f"Files saved: results_summary.csv, results_panels.csv, dm_tests.csv, "
      f"test_losses_per_date.csv, forecast_dump.csv, "
      f"fig_predicted_vs_realized.png, fig_qlike_bars.png, fig_cumulative_loss_diff.png")

In [ ]:
for (index_name, estimator), tables in panel_tables.items():
    safe_name = f"{index_name}_{estimator}".replace(" ", "")
    tables["QLIKE"].to_latex(f"table_qlike_{safe_name}.tex")
    tables["RMSE"].to_latex(f"table_rmse_{safe_name}.tex")
    tables["MAE"].to_latex(f"table_mae_{safe_name}.tex")

print("Saved 18 .tex tables (9 panels × 3 metrics).")